In [3]:
## import the necessary packages
import numpy as np
import os
from scipy.optimize import linear_sum_assignment
import csv
import pandas as pd
from tqdm import tqdm
import tifffile
from math import*

In [19]:
method = 'GNN'
#pred_dir = r'../results/results_real_annotated_2/trial1/Results_0_constraints_threshold' #directory containing the predicted nucleus and golgi centroids
pred_dir = r'../results/results_real_annotated_1/trial1/Results_0_constraints'
pred_dir = r'../results/results_real_annotated_1/trial1/Results_0'
gt_dir = r"../data/vectors" #directory with the ground truth vectors
img_dir = r"../data/images"#directory with images

In [20]:
nuclei_thresholds = [8.8]
golgi_thresholds = [4.4]
levels_ = [0]
lvl_ = 0

numbers_ = [0, 1, 2, 3, 4, 5, 6, 7] #crops

## name of the images
imgs = ['Crop1', 'Crop2', 'Crop3', 'Crop4', 'Crop5_BC', 'Crop6_BC', 'Crop7_BC','Crop8_BC']

image_dimensions = [[0.333,0.333,0.270], [0.333,0.333,0.270], [0.333,0.333,0.270], [0.333,0.333,0.270],
              [0.333,0.333,0.270], [0.333,0.333,0.270], [0.333,0.333,0.400], [0.333,0.333,0.400]] #um

info = ['test']*8

performance_metrics = pd.DataFrame(columns = ["Image", "Method", "Type", "NucleusTh", "GolgiTh", "Threshold_level",
                                              "CosineSimilarityM",
                                              "CosineSimilaritySTD", "VecErrorM","VecErrorSTD",
                                              "DistanceNuM", "DistanceNuSTD", "DistanceGoM",
                                              "DistanceGoSTD", "TP", "FP", "FN", "TPR", "FPR", "FNR"])

metrics_stats = pd.DataFrame(columns = ["Image", "Method", "Type", "NucleusTh", "GolgiTh", "Threshold_level",
                                        "cosine similarity", "vec_error", "nuclei", "golgi"])

allmetrics = pd.DataFrame(columns = ["Image", "Method", "Type", "NucleusTh", "GolgiTh", "Threshold_level",
                                      "index_tp_gt", "cosine similarity", "vec_error", "nuclei", "golgi"])

In [21]:
''' Define the metrics ''' 
def square_rooted(x):
    return round(np.sqrt(sum([a*a for a in x])),3)
 
def cosine_similarity(x,y):
    numerator = sum(a*b for a,b in zip(x,y))
    denominator = square_rooted(x)*square_rooted(y)
    return round(numerator/float(denominator),3)

#Euclidean distance computed in um
def distance_um(p, q, dimx, dimy, dimz):
    dist_um = (((p[0]-q[0])*dimx)**2)+(((p[1]-q[1])*dimy)**2)+(((p[2]-q[2])*dimz)**2)
    return np.sqrt(dist_um) 

#ignore the borders of the image
def inside_img(coord,img_dim_x,img_dim_y,img_dim_z,x_y_lim,z_lim):
    return coord[0]<img_dim_x-x_y_lim and coord[0]>x_y_lim and coord[1]<img_dim_y-x_y_lim and coord[1]>x_y_lim and coord[2]<img_dim_z-z_lim and coord[2]>0

In [22]:
for image_nb in tqdm(numbers_):
    
    pred_vectors = os.path.join(pred_dir, imgs[image_nb] + '.csv')
    gt_vectors = os.path.join(gt_dir, imgs[image_nb] + '.csv')

    ## read the image and get its dimensions
    image = tifffile.imread(os.path.join(img_dir, imgs[image_nb] + '.tif'))
    (img_dim_x, img_dim_y, img_dim_z, channels) = np.shape(image)

    #voxel's physical dimensions
    x_spacing = image_dimensions[image_nb][0]
    y_spacing = image_dimensions[image_nb][1]
    z_spacing = image_dimensions[image_nb][2]
    
    #limits to ignore vectors at the borders of the image
    x_y_lim = int(7/x_spacing)  #(voxels)  16
    z_lim = int(5/z_spacing)    #(voxels)  5

    print('Reading the csv file with the ground truth vectors')
    ## nuclei and golgi centroids
    nuclei_centroids_gt = [] 
    golgi_centroids_gt = []
    
    #open the csv file and save the gt nucleus and Golgi centroids
    file = open(gt_vectors, "rU")
    reader = csv.reader(file, delimiter=';')
    for row in reader:
        if row[0] != 'YN,XN,ZN,YG,XG,ZG':
            aux = row[0].split(",")
            YN = int(float(aux[0]))-1
            XN = int(float(aux[1]))-1
            ZN = int(float(aux[4]))-1
            YG = int(float(aux[2]))-1
            XG = int(float(aux[3]))-1
            ZG = int(float(aux[5]))-1
            
            if inside_img(np.asarray([XN,YN,ZN]), img_dim_x, img_dim_y, img_dim_z, x_y_lim, z_lim) and inside_img(np.asarray([XG,YG,ZG]), img_dim_x,img_dim_y,img_dim_z,x_y_lim,z_lim):
                nuclei_centroids_gt.append((XN,YN,ZN))
                golgi_centroids_gt.append((XG,YG,ZG))     
    
    golgi_centroids_gt = np.asarray(golgi_centroids_gt)
    nuclei_centroids_gt = np.asarray(nuclei_centroids_gt)
    
    #Remove predicted nuclei and golgi at image borders
    n_centroids = []
    g_centroids = []
    #open the csv file and save the gt nucleus and Golgi centroids
    file = open(pred_vectors, "rU")
    reader = csv.reader(file, delimiter=';')
    for row in reader:
        if row[0] != 'YN,XN,ZN,YG,XG,ZG':
            aux = row[0].split(",")
            YN = int(float(aux[0]))-1
            XN = int(float(aux[1]))-1
            ZN = int(float(aux[4]))-1
            YG = int(float(aux[2]))-1
            XG = int(float(aux[3]))-1
            ZG = int(float(aux[5]))-1
            
            if inside_img(np.asarray([XN,YN,ZN]), img_dim_x, img_dim_y, img_dim_z, x_y_lim, z_lim) and inside_img(np.asarray([XG,YG,ZG]), img_dim_x,img_dim_y,img_dim_z,x_y_lim,z_lim):
                if distance_um([XN,YN,ZN], [XG,YG,ZG], x_spacing, y_spacing, z_spacing)<18:
                    n_centroids.append((XN,YN,ZN))
                    g_centroids.append((XG,YG,ZG))     
            
    nuclei_centroids = np.asarray(n_centroids)
    golgi_centroids = np.asarray(g_centroids)
    
    print('Evaluation')
    ''' Assignment nuclei centroids '''
    ## compute the Euclidean distance between the predicted and ground truth centroids
    matrix = np.zeros((len(nuclei_centroids),len(nuclei_centroids_gt)))
    
    ## build the cost matrix
    for i in range(0,len(nuclei_centroids)):
        for j in range(0,len(nuclei_centroids_gt)):
            matrix[i,j] = distance_um(nuclei_centroids[i], nuclei_centroids_gt[j], x_spacing, y_spacing, z_spacing) + distance_um(golgi_centroids[i], golgi_centroids_gt[j], x_spacing, y_spacing, z_spacing)
    
    matrix[matrix>10] = 2000
    
    ## method to solve the linear assignment problem
    row_ind, col_ind = linear_sum_assignment(matrix)
    
    ''' Compute the metrics for the vectors '''
    for n_th, g_th, thlvl in zip(nuclei_thresholds, golgi_thresholds, levels_):
        metrics = pd.DataFrame(columns = ["Image", "Method", "Type", "NucleusTh", "GolgiTh", "Threshold_level",
                                      "cosine similarity", "vec_error", "nuclei", "golgi"])

        if thlvl==lvl_:
            index_tp = []  ## positions in vectors nuclei_centroids, golgi_centroids, that are
                            ## true positives
                            
            index_tp_gt = [] ## positions in vectors nuclei_centroids_gt and golgi_centroids_gt,
                              ## that correspond to true positives

        for i in range(0, len(row_ind)):
            n_coord = nuclei_centroids[row_ind[i]]
            g_coord = golgi_centroids[row_ind[i]]
        
            vec = g_coord - n_coord
        
            n_coord_gt = nuclei_centroids_gt[col_ind[i]]
            g_coord_gt = golgi_centroids_gt[col_ind[i]]
        
            vec_gt = g_coord_gt - n_coord_gt
            
            dist_n_centroids = distance_um(n_coord, n_coord_gt, x_spacing, y_spacing, z_spacing)
            dist_g_centroids = distance_um(g_coord, g_coord_gt, x_spacing, y_spacing, z_spacing)
            vec_error = distance_um(vec, vec_gt, x_spacing, y_spacing, z_spacing)
            
            cos_sim = cosine_similarity(vec, vec_gt)
            
            if dist_n_centroids<=n_th and dist_g_centroids<=g_th:

                res = {"Image": imgs[image_nb], "Method": method, "Type": info[image_nb], "NucleusTh": n_th, "GolgiTh": g_th,
                       "Threshold_level": thlvl,
                       "cosine similarity": abs(cos_sim), "vec_error": vec_error, 
                       "nuclei": dist_n_centroids, "golgi": dist_g_centroids}
                
                res_aux = {"Image": imgs[image_nb], "Method": method, "Type": info[image_nb], "NucleusTh": n_th, "GolgiTh": g_th,
                       "Threshold_level": thlvl, "index_tp_gt": col_ind[i],
                       "cosine similarity": abs(cos_sim), "vec_error": vec_error, 
                       "nuclei": dist_n_centroids, "golgi": dist_g_centroids}
                
                row_aux = len(allmetrics)
                allmetrics.loc[row_aux] = res_aux
                
                row = len(metrics)
                metrics.loc[row] = res
                
                row_stats = len(metrics_stats)
                metrics_stats.loc[row_stats] = res
                
                if thlvl==lvl_:
                    index_tp.append(row_ind[i])
                    index_tp_gt.append(col_ind[i])

        print(metrics.Image.unique())
        numeric_metrics = metrics.select_dtypes(include='number')
        # Calculate mean and standard deviation
        metrics_mean = numeric_metrics.mean()
        metrics_std = numeric_metrics.std()
        TP = len(metrics)
        
        FP = np.shape(golgi_centroids)[0] - len(metrics)
        
        FN = np.shape(golgi_centroids_gt)[0] - len(metrics)
        
        TPR = TP/(TP+FN)
        
        FPR = FP/(FP+TP)
        
        FNR = FN/(FN+TP)
        
        res = {"Image": imgs[image_nb], "Method": method, "Type": info[image_nb], 
               "NucleusTh": n_th, "GolgiTh": g_th, "Threshold_level": thlvl,
               "CosineSimilarityM": metrics_mean['cosine similarity'],
               "CosineSimilaritySTD": metrics_std['cosine similarity'], 
               "VecErrorM": metrics_mean['vec_error'],
               "VecErrorSTD": metrics_std['vec_error'],
               "DistanceNuM": metrics_mean['nuclei'], 
               "DistanceNuSTD": metrics_std['nuclei'], 
               "DistanceGoM": metrics_mean['golgi'], 
               "DistanceGoSTD": metrics_std['golgi'], 
               "TP": TP, 
               "FP": FP, 
               "FN": FN, 
               "TPR": TPR, 
               "FPR": FPR,
               "FNR": FNR}
        
        row = len(performance_metrics)
        performance_metrics.loc[row] = res

  0%|          | 0/8 [00:00<?, ?it/s]

Reading the csv file with the ground truth vectors
Evaluation


 12%|█▎        | 1/8 [00:00<00:02,  2.93it/s]

['Crop1']
Reading the csv file with the ground truth vectors
Evaluation


 25%|██▌       | 2/8 [00:00<00:02,  2.52it/s]

...

['Crop5_BC']
Reading the csv file with the ground truth vectors
Evaluation


 75%|███████▌  | 6/8 [00:01<00:00,  4.08it/s]

['Crop6_BC']
Reading the csv file with the ground truth vectors
Evaluation


100%|██████████| 8/8 [00:02<00:00,  3.54it/s]

['Crop7_BC']
Reading the csv file with the ground truth vectors
Evaluation
['Crop8_BC']


In [23]:
final_metrics = performance_metrics.groupby(["Threshold_level"], as_index=False).agg({'CosineSimilarityM': np.mean,
                                                 "CosineSimilaritySTD": np.mean,
                                                 "VecErrorM": np.mean,
                                                 "VecErrorSTD": np.mean,
                                                 "DistanceNuM": np.mean, 
                                                 "DistanceNuSTD": np.mean, 
                                                 "DistanceGoM": np.mean, 
                                                 "DistanceGoSTD": np.mean, 
                                                 "TP": np.sum, 
                                                 "FP": np.sum, 
                                                 "FN": np.sum, 
                                                 "TPR": np.mean, 
                                                 "FPR": np.mean,
                                                 "FNR": np.mean})

C:\Users\001149822\AppData\Local\Temp\ipykernel_3340\262267097.py:1: FutureWarning: The provided callable <function mean at 0x000002A143F18700> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  final_metrics = performance_metrics.groupby(["Threshold_level"], as_index=False).agg({'CosineSimilarityM': np.mean,
C:\Users\001149822\AppData\Local\Temp\ipykernel_3340\262267097.py:1: FutureWarning: The provided callable <function sum at 0x000002A143F14790> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  final_metrics = performance_metrics.groupby(["Threshold_level"], as_index=False).agg({'CosineSimilarityM': np.mean,


In [24]:
final_metrics

,Threshold_level,CosineSimilarityM,CosineSimilaritySTD,VecErrorM,VecErrorSTD,DistanceNuM,DistanceNuSTD,DistanceGoM,DistanceGoSTD,TP,FP,FN,TPR,FPR,FNR
0,0,0.997094,0.017631,0.063459,0.323448,0.622227,0.303182,0.565568,0.06499,389,122,19,0.94837,0.230805,0.05163
